In [4]:
import pandas as pd
import numpy as np
import csv
import os
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from pyproj import Transformer

warnings.filterwarnings('ignore')

# 1. CARGA DE DATOS (Universal)
ruta_entrada = 'accidentes_con_trafico_final.csv'

if not os.path.exists(ruta_entrada):
    try:
        from google.colab import files
        print("Entorno Colab detectado. Por favor, sube el archivo original:")
        uploaded = files.upload()
    except ImportError:
        print(f"❌ Error: No se encuentra '{ruta_entrada}' en la carpeta local.")

# 2. Lector para corregir errores de formato en el CSV
def lector_profesional(path):
    datos = []
    columnas = ["fecha","hora","dia_semana","distrito","num_expediente","tipo_accidente",
               "tipo_vehiculo","sexo","rango_edad","estado_meteorologico",
               "coordenada_x_utm","coordenada_y_utm","direccion_unica","es_festivo",
               "id_sensor_cercano","intensidad","ocupacion","vmed"]
    with open(path, 'r', encoding='latin1') as f:
        reader = csv.reader(f)
        next(reader) 
        for row in reader:
            if len(row) == 18:
                datos.append(row)
            elif len(row) > 18:
                inicio, final = row[:12], row[-5:]
                dir_fix = " ".join(row[12:-5])
                datos.append(inicio + [dir_fix] + final)
    return pd.DataFrame(datos, columns=columnas)

print("⏳ Cargando y limpiando base de datos...")
df = lector_profesional(ruta_entrada)

⏳ Cargando y limpiando base de datos...


In [5]:
# 1. Conversión de tipos
df['coordenada_x_utm'] = pd.to_numeric(df['coordenada_x_utm'], errors='coerce')
df['coordenada_y_utm'] = pd.to_numeric(df['coordenada_y_utm'], errors='coerce')
df['es_festivo'] = pd.to_numeric(df['es_festivo'], errors='coerce').fillna(0)
df['fecha'] = pd.to_datetime(df['fecha'], format='%Y-%m-%d', errors='coerce')
df = df.dropna(subset=['fecha'])
df['mes_dia'] = df['fecha'].dt.strftime('%m-%d')
df['dia_semana'] = df['dia_semana'].str.lower().str.strip()

# 2. Rescate de coordenadas (Imputación por calle)
coords_calle = df.groupby('direccion_unica')[['coordenada_x_utm', 'coordenada_y_utm']].transform('mean')
df['coordenada_x_utm'] = df['coordenada_x_utm'].fillna(coords_calle['coordenada_x_utm'])
df['coordenada_y_utm'] = df['coordenada_y_utm'].fillna(coords_calle['coordenada_y_utm'])

df_madrid = df.dropna(subset=['coordenada_x_utm', 'coordenada_y_utm']).copy()

# 3. Conversión a GPS (Filtro estricto para Madrid)
transformer = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)
lons, lats = transformer.transform(df_madrid['coordenada_x_utm'].values, df_madrid['coordenada_y_utm'].values)
df_madrid['lat'], df_madrid['lon'] = lats, lons

df_madrid = df_madrid[(df_madrid['lat'] > 40.3) & (df_madrid['lat'] < 40.6) & 
                      (df_madrid['lon'] > -3.9) & (df_madrid['lon'] < -3.5)]

print(f"✅ Limpieza terminada. Registros útiles: {len(df_madrid)}")

✅ Limpieza terminada. Registros útiles: 152815


In [6]:
# 1. Definición de Slots de 30 min y Sugerencias
def slot_30min(hora_str):
    try:
        h, m = map(int, str(hora_str).split(':')[:2])
        return f"{h:02d}:{'00' if m < 30 else '30'}"
    except: return "00:00"

def sugerencia_label(hora_str):
    try:
        h = int(str(hora_str).split(':')[0])
        if 0 <= h < 7: return 'Sugerencia: Madrugada (0-7h)'
        elif 7 <= h < 9: return 'Sugerencia: Punta Entrada (7-9h)'
        elif 9 <= h < 15: return 'Sugerencia: Mañana Trabajo (9-15h)'
        elif 15 <= h < 17: return 'Sugerencia: Salida Comida (15-17h)'
        elif 17 <= h < 21: return 'Sugerencia: Tarde Ocio (17-21h)'
        else: return 'Sugerencia: Noche (21-0h)'
    except: return 'Desconocido'

df_madrid['slot_30m'] = df_madrid['hora'].apply(slot_30min)
df_madrid['sugerencia_label'] = df_madrid['hora'].apply(sugerencia_label)

# 2. Función de Cálculo de Riesgo (Log + Z-Score + MinMax)
def calcular_riesgo_tecnico(df_input, cols):
    agg = df_input.groupby(cols).size().reset_index(name='n_acc')
    agg['log_n'] = np.log1p(agg['n_acc'])
    agg['z'] = StandardScaler().fit_transform(agg[['log_n']])
    agg['riesgo'] = MinMaxScaler(feature_range=(0, 10)).fit_transform(agg[['z']]).round(2)
    return agg

r_cal = calcular_riesgo_tecnico(df_madrid, ['direccion_unica', 'mes_dia', 'slot_30m'])
r_sem = calcular_riesgo_tecnico(df_madrid[df_madrid['es_festivo'] == 0], ['direccion_unica', 'dia_semana', 'slot_30m'])

# 3. Unificación (Limpieza de duplicados para evitar el TypeError)
for col in ['r_sem', 'r_cal', 'riesgo_total']:
    if col in df_madrid.columns: df_madrid.drop(columns=col, inplace=True)

df_madrid = df_madrid.merge(r_sem[['direccion_unica', 'dia_semana', 'slot_30m', 'riesgo']], on=['direccion_unica', 'dia_semana', 'slot_30m'], how='left').rename(columns={'riesgo': 'r_sem'})
df_madrid = df_madrid.merge(r_cal[['direccion_unica', 'mes_dia', 'slot_30m', 'riesgo']], on=['direccion_unica', 'mes_dia', 'slot_30m'], how='left').rename(columns={'riesgo': 'r_cal'})

# Media unificada ignorando NaNs
df_madrid['riesgo_total'] = df_madrid[['r_sem', 'r_cal']].mean(axis=1).fillna(0).round(2)

print("🚀 Riesgo IRU (30 min) generado correctamente.")

🚀 Riesgo IRU (30 min) generado correctamente.


In [7]:
import ipywidgets as widgets
from ipywidgets import interact
import folium
import branca.colormap as cm

colormap = cm.LinearColormap(colors=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'], vmin=0, vmax=10)
slots_lista = sorted(df_madrid['slot_30m'].unique().tolist())

@interact(modo=['Día de la Semana', 'Fecha Específica'])
def visualizador_final(modo):
    opc_t = ['lunes', 'martes', 'miercoles', 'jueves', 'viernes', 'sabado', 'domingo'] if modo == 'Día de la Semana' else sorted(df_madrid['mes_dia'].unique().tolist())
    
    @interact(seleccion=widgets.Dropdown(options=opc_t, description='Día/Fecha:'),
              h_inicio=widgets.Dropdown(options=slots_lista, value='08:00', description='Desde:'),
              h_fin=widgets.Dropdown(options=slots_lista, value='10:00', description='Hasta:'))
    def plot_mapa(seleccion, h_inicio, h_fin):
        if h_inicio > h_fin: return print("❌ Error: 'Desde' debe ser anterior a 'Hasta'.")
            
        f_dia = (df_madrid['dia_semana'] == seleccion) if modo == 'Día de la Semana' else (df_madrid['mes_dia'] == seleccion)
        f_tiempo = (df_madrid['slot_30m'] >= h_inicio) & (df_madrid['slot_30m'] <= h_fin)
        
        data = df_madrid[f_dia & f_tiempo].groupby(['direccion_unica', 'lat', 'lon'])['riesgo_total'].mean().reset_index()

        m = folium.Map(location=[40.4167, -3.7033], zoom_start=12, tiles='cartodbpositron')
        colormap.add_to(m)
        for _, r in data.iterrows():
            folium.CircleMarker([r['lat'], r['lon']], radius=5, color=colormap(r['riesgo_total']), fill=True, 
                               tooltip=f"<b>{r['direccion_unica']}</b><br>Riesgo Medio: {r['riesgo_total']:.2f}").add_to(m)
        display(m)

interactive(children=(Dropdown(description='modo', options=('Día de la Semana', 'Fecha Específica'), value='Dí…

In [8]:
# Guardamos solo lo esencial para que el archivo no sea gigante
columnas_ia = ['fecha', 'dia_semana', 'slot_30m', 'sugerencia_label', 'distrito', 'direccion_unica', 'lat', 'lon', 'riesgo_total']
df_madrid[columnas_ia].to_csv('madrid_minable_view_final.csv', index=False)
print("✅ Archivo 'madrid_minable_view_final.csv' listo para compartir.")

✅ Archivo 'madrid_minable_view_final.csv' listo para compartir.
